## Stage 0: Signal Extraction & Preprocessing

- Due to data confidentiality and IP rights on raw data format, the data extraction procedure (which is irrelevant from the paper) from the raw files cannot be disclosed.
- We have provided the code that was used to extract the signals from R,G,B values, followed by bandpass and detrending — as described in the manuscript.
- The raw data cannot be disclosed, thus, this part cannot be reproduced.

In [ ]:
import numpy as np
import pandas as pd

from processing import process_all_signals


def process_one_recording(patient_id: str, timestamps: np.ndarray, bgr_data: np.ndarray, fps: int = 30) -> pd.DataFrame:
    """
    timestamps: 1D array, one timestamp per sample (already loaded by Stage 1)
    bgr_data:   (N, 3) array of [B, G, R] intensities (already loaded by Stage 1)

    Returns a long-format DataFrame: one row per sample for every channel.
    """
    sigB, sigG, sigR = bgr_data[:, 0], bgr_data[:, 1], bgr_data[:, 2]

    df_signals = process_all_signals(sigR, sigG, sigB, fps=fps)
    if df_signals is None or df_signals.empty:
        return None

    min_len = min(len(timestamps), len(df_signals))
    df_signals = df_signals.iloc[:min_len].copy()
    df_signals.insert(0, "Timestamps", timestamps[:min_len])
    df_signals.insert(0, "Patient_ID", patient_id)
    return df_signals


def process_recording_set(loaded_recordings: dict) -> pd.DataFrame:
    """
    loaded_recordings: {patient_id: (timestamps, bgr_data)} -- already produced by Stage 1. 
    """
    rows = []
    for patient_id, (timestamps, bgr_data) in loaded_recordings.items():
        df = process_one_recording(patient_id, timestamps, bgr_data)
        if df is not None:
            rows.append(df)

    results_df = pd.concat(rows, ignore_index=True)
    results_df["time_cumulative"] = (
        results_df.groupby("Patient_ID")["Timestamps"].transform(lambda t: (t - t.min()) / 1e6)
    )
    return results_df


# ----------------------------------------------------------------------
# AFTER THIS DATA EXTRACTION STAGE:
#
#   1. results_df is reshaped to wide format (one row per Patient_ID,
#      each signal channel as a list-valued column).
#   2. The result is merged with demographics_fpg_df on Patient_ID to
#      produce merged_df (=dataset_df.parquet)-- the saved checkpoint that the rest of the
#      shared pipeline (tierA cropping, SQI filtering, TF-Map generation, CV training) starts from.
#
# ----------------------------------------------------------------------